In [27]:
%pip install requests beautifulsoup4 pandas lxml 


Note: you may need to restart the kernel to use updated packages.


In [30]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [31]:

# --- PART 1: SCRAPING ---
url = "https://books.toscrape.com/"
headers = {"User-Agent": "IT-Student-Lab-Extraction"}

print("Connecting to website...")
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "lxml")

books_data = []
articles = soup.find_all('article', class_='product_pod')

for article in articles:
    title = article.h3.a['title']
    price_raw = article.find('p', class_='price_color').text
    # REGEX: Keeps only numbers and dots
    price_clean = re.sub(r'[^\d.]', '', price_raw) 
    
    books_data.append({
        "Item_Name": title,
        "Price_GBP": float(price_clean), # Ensure this is a float
        "Status": article.find('p', class_='instock availability').text.strip()
    })


Connecting to website...


In [32]:

# --- PART 2: THE SAFETY CHECK ---
if not books_data:
    print("❌ Error: No data was scraped. Check your internet connection or the URL.")
else:
    df_raw = pd.DataFrame(books_data)
    print(f"✅ Scraped {len(df_raw)} items successfully.")

    # --- PART 3: PREPROCESSING ---
    # 1. Feature Engineering
    df_raw['Price_Level'] = df_raw['Price_GBP'].apply(lambda x: 'Premium' if x > 30 else 'Standard')

    # 2. Encoding
    le = LabelEncoder()
    df_raw['Status_Encoded'] = le.fit_transform(df_raw['Status'])
    df_raw['Level_Encoded'] = le.fit_transform(df_raw['Price_Level'])

    # 3. Scaling
    scaler = StandardScaler()
    df_raw['Price_Scaled'] = scaler.fit_transform(df_raw[['Price_GBP']])

    print("\n--- Final Preprocessed Data (Top 5) ---")
    print(df_raw[['Item_Name', 'Price_GBP', 'Price_Scaled', 'Level_Encoded']].head())

    # Save to your local folder
    df_raw.to_csv('terfi_lab3_final.csv', index=False)

✅ Scraped 20 items successfully.

--- Final Preprocessed Data (Top 5) ---
                               Item_Name  Price_GBP  Price_Scaled  \
0                   A Light in the Attic      51.77      0.930145   
1                     Tipping the Velvet      53.74      1.063686   
2                             Soumission      50.10      0.816940   
3                          Sharp Objects      47.82      0.662385   
4  Sapiens: A Brief History of Humankind      54.23      1.096902   

   Level_Encoded  
0              0  
1              0  
2              0  
3              0  
4              0  
